# Notebook 01: Data Understanding and Exploratory Data Analysis

## Overview
This is the first notebook of the machine learning track for lung and colon cancer histopathology classification. It uses the LC25000 dataset, which holds 25,000 colour images across five tissue classes. Before any feature is extracted or any model is trained, the data is inspected so that later decisions rest on real evidence rather than assumption. This notebook looks at how many images each class has, what the images look like, how colour and texture differ across classes, and what the Macenko stain normalisation does to an image.

## Objectives
1. List every image and confirm the class balance of the dataset.
2. Display sample images from each of the five classes.
3. Inspect basic image properties such as size and channel count.
4. Explore how colour and texture differ between the tissue classes.
5. Show the effect of Macenko stain normalisation on a sample image.

## Dataset
LC25000, Borkowski et al. (2019), arXiv:1912.12142. Source: https://github.com/tampapath/lung_colon_image_set
Classes: colon_aca, colon_n, lung_aca, lung_n, lung_scc, with 5,000 images each.

In [ ]:
# ----- Environment setup -----
!pip install -q scikit-image seaborn joblib

from google.colab import drive
drive.mount('/content/drive')

import os, sys

# ---- Clone the project repository so notebooks can import from src/ ----
REPO       = "lung-colon-cancer-histopathology-ml"
REPO_LOCAL = "/content/" + REPO
REPO_URL   = "https://github.com/hodyek/" + REPO + ".git"

if not os.path.exists(REPO_LOCAL):
    !git clone $REPO_URL $REPO_LOCAL

# Insert at position 0 so our src/ always takes priority over any Colab defaults.
if REPO_LOCAL not in sys.path:
    sys.path.insert(0, REPO_LOCAL)

# Quick sanity-check that the import will work before running any cells.
import importlib.util
_spec = importlib.util.find_spec("src.dataset")
if _spec is None:
    raise ImportError(
        "src.dataset not found. Check that the clone succeeded and "
        "that src/__init__.py exists in the repository."
    )
print("src modules found at:", os.path.join(REPO_LOCAL, "src"))

# ---- Project folders on Google Drive ----
DRIVE_ROOT = "/content/drive/MyDrive/" + REPO
DATA_DIR   = os.path.join(DRIVE_ROOT, "data", "lung_colon_image_set")
FIG_DIR    = os.path.join(DRIVE_ROOT, "figures")
MODEL_DIR  = os.path.join(DRIVE_ROOT, "models")
FEAT_DIR   = os.path.join(DRIVE_ROOT, "features")
for d in (FIG_DIR, MODEL_DIR, FEAT_DIR):
    os.makedirs(d, exist_ok=True)

CLASSES = ["colon_aca", "colon_n", "lung_aca", "lung_n", "lung_scc"]
print("Setup complete. CLASSES:", CLASSES)


In [ ]:
# Build a table of every image with its class label.
import pandas as pd
from src.dataset import list_images, CLASSES

df = list_images(DATA_DIR)
print("Total images found:", len(df))
counts = df["label"].value_counts().reindex(CLASSES)
print(counts)

import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
counts.plot(kind="bar", color="steelblue")
plt.title("Number of images per class")
plt.ylabel("Image count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "01_class_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

The dataset returns 25,000 images in total and the bar chart shows each of the five classes holds the same 5,000 images. This means the dataset is perfectly balanced, so accuracy is a fair headline metric and no class weighting is needed. A balanced dataset also makes the confusion matrix easy to read later, since every row holds the same number of true samples. If the count had come out uneven, the plan would have changed to include resampling or class weights.

In [ ]:
# Show a few sample images from each class to see what the model will work with.
import matplotlib.pyplot as plt
from src.dataset import load_image

fig, axes = plt.subplots(len(CLASSES), 4, figsize=(11, 13))
for r, cls in enumerate(CLASSES):
    paths = df[df["label"] == cls]["path"].head(4).tolist()
    for col, p in enumerate(paths):
        axes[r, col].imshow(load_image(p))
        axes[r, col].axis("off")
        if col == 0:
            axes[r, col].set_ylabel(cls, rotation=0, labelpad=45, fontsize=11)
    axes[r, 0].set_title(cls, loc="left", fontsize=11)
plt.suptitle("Sample images from each class", y=0.99)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "01_sample_images.png"), dpi=150, bbox_inches="tight")
plt.show()

The samples make the visual differences between classes clear. The benign tissue classes look more regular and ordered, while the cancer classes show crowded and irregular structures. Colon and lung tissue have different overall shapes, and the two lung cancer classes look fairly similar to each other. This early similarity between lung_aca and lung_scc is worth remembering, because it is the most likely place the model will make mistakes.

In [ ]:
# Check the raw size and channel count of a few images.
import numpy as np
import cv2

sample_paths = df["path"].sample(6, random_state=42).tolist()
for p in sample_paths:
    raw = cv2.imread(p)
    print(os.path.basename(p), "shape:", raw.shape, "dtype:", raw.dtype)

Every sampled image comes back at 768 by 768 pixels with three colour channels and 8 bit values. The images are already a consistent size, so resizing is only needed to make feature extraction faster and to match the deep learning pipeline. Working at full resolution would slow the texture features down a lot for little gain. For that reason the rest of the project resizes images to 224 by 224 before extracting features.

In [ ]:
# Explore how average colour differs between classes, using a sample for speed.
import numpy as np
from src.dataset import load_image

rows = []
for cls in CLASSES:
    paths = df[df["label"] == cls]["path"].sample(60, random_state=0).tolist()
    means = np.array([load_image(p).reshape(-1, 3).mean(axis=0) for p in paths]).mean(axis=0)
    rows.append({"class": cls, "R": means[0], "G": means[1], "B": means[2]})
colour_df = pd.DataFrame(rows).set_index("class")
print(colour_df.round(1))

colour_df.plot(kind="bar", figsize=(8, 4), color=["#c0392b", "#27ae60", "#2980b9"])
plt.title("Average RGB value per class (sample of 60 images each)")
plt.ylabel("Mean intensity")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "01_mean_rgb_per_class.png"), dpi=150, bbox_inches="tight")
plt.show()

The colour averages separate the colon and lung tissue groups quite well, which fits the pink and purple staining seen in the samples. The differences between classes are real but not huge, so colour alone will probably not be enough to tell every class apart. This is the reason the pipeline also adds texture and edge features later. The plot supports the choice of a multi family feature set rather than relying on a single descriptor.

In [ ]:
# Texture preview: show the Local Binary Pattern map next to the grayscale image.
import numpy as np
from skimage.feature import local_binary_pattern
from skimage.color import rgb2gray
from src.dataset import load_image

fig, axes = plt.subplots(len(CLASSES), 2, figsize=(6, 14))
for r, cls in enumerate(CLASSES):
    p = df[df["label"] == cls]["path"].iloc[0]
    img = load_image(p)
    gray = (rgb2gray(img.astype(np.float32) / 255.0) * 255).astype(np.uint8)
    lbp = local_binary_pattern(gray, 8, 1, method="uniform")
    axes[r, 0].imshow(gray, cmap="gray"); axes[r, 0].set_title(cls + " grayscale", fontsize=9); axes[r, 0].axis("off")
    axes[r, 1].imshow(lbp, cmap="gray"); axes[r, 1].set_title(cls + " LBP texture", fontsize=9); axes[r, 1].axis("off")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "01_texture_preview.png"), dpi=150, bbox_inches="tight")
plt.show()

The Local Binary Pattern maps highlight the fine texture of each tissue type and remove the colour information. The cancer classes produce busier texture maps because their cells are crowded and irregular, while benign tissue looks smoother. This shows that texture carries useful signal that colour does not, which supports including GLCM and LBP features. Seeing the texture directly also builds confidence that the handcrafted features describe something real in the tissue.

In [ ]:
# Show what Macenko stain normalisation does to one image.
from src.dataset import load_image, macenko_normalize

p = df[df["label"] == "lung_aca"]["path"].iloc[0]
orig = load_image(p)
norm = macenko_normalize(orig)

fig, ax = plt.subplots(1, 2, figsize=(9, 5))
ax[0].imshow(orig); ax[0].set_title("Original"); ax[0].axis("off")
ax[1].imshow(norm); ax[1].set_title("Macenko normalised"); ax[1].axis("off")
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, "01_macenko_before_after.png"), dpi=150, bbox_inches="tight")
plt.show()

After Macenko normalisation the stain colours are mapped onto a fixed reference, so the pink and purple tones look more standard. The structure of the tissue stays the same while the colour becomes more consistent. This matters because the colour features would otherwise pick up differences in staining and scanning rather than real differences in tissue. Normalising the stain first should make the colour features more honest about the biology.

## Summary
The dataset is balanced with 5,000 images in each of the five classes, all sized 768 by 768. The classes differ in both colour and texture, but the two lung cancer classes look similar and are the likely source of future errors. Colour alone is not strongly separating, which justifies a feature set that combines colour, texture, and edge structure. Macenko normalisation makes stain colour more consistent without changing tissue shape. The next notebook splits the data and turns these images into a feature table.